# Lab 2 — Digital Image Fundamentals
**Course:** ARTI407 – Image Processing  
**College of Computer Science and Information Technology**  
**Imam Abdulrahman Bin Faisal University**

This notebook covers the lab manual:
1. **Sampling and Quantization**
2. **Arithmetic Operations** (add, subtract, add constant)
3. **Set / Logical Operations** (union, intersection, difference, symmetric difference)
4. **Practical Tasks** (all solved)

> **Note on images:** The lab references `lena_gray_256.tif`, `cameraman.tif`, `A.png`, `B.png`. Since those files aren't bundled with the notebook, this notebook uses built-in `skimage` images so every cell runs out of the box. To use your own image, just replace the loading line.

## Setup

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage import data

%matplotlib inline

---
## Section 1 — Image Sampling and Quantization

- **Sampling** = how many pixels we keep (spatial resolution).
- **Quantization** = how many gray levels we keep (intensity resolution).

In [ ]:
def sample_image(image, factor):
    """Downsamples the image by the given factor."""
    height, width = image.shape[:2]
    sampled_image = cv2.resize(image, (width // factor, height // factor),
                               interpolation=cv2.INTER_NEAREST)
    return sampled_image

def quantize_image(image, levels):
    """Reduces the number of grayscale levels in the image."""
    quantized_image = np.floor(image / (256 // levels)) * (256 // levels)
    quantized_image = quantized_image.astype(np.uint8)
    return quantized_image

def plot_images(original, sampled, quantized):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1), plt.imshow(original, cmap='gray')
    plt.title('Original Image'), plt.axis('off')
    plt.subplot(1, 3, 2), plt.imshow(sampled, cmap='gray')
    plt.title('Sampled Image'), plt.axis('off')
    plt.subplot(1, 3, 3), plt.imshow(quantized, cmap='gray')
    plt.title('Quantized Image'), plt.axis('off')
    plt.show()

In [ ]:
original_image = data.camera()

sampling_factor = 14
quantization_levels = 9

sampled_image = sample_image(original_image, sampling_factor)
quantized_image = quantize_image(original_image, quantization_levels)

plot_images(original_image, sampled_image, quantized_image)

---
## Section 2 — Arithmetic Operations

Add two images by converting them to numpy arrays and adding element-wise.

In [ ]:
img1 = Image.fromarray(data.camera())
img2 = Image.fromarray(data.moon())

resize = (400, 400)
img1 = img1.resize(resize, Image.Resampling.LANCZOS)
img2 = img2.resize(resize, Image.Resampling.LANCZOS)

im1arr = np.asarray(img1)
im2arr = np.asarray(img2)

addition = im1arr + im2arr   # uint8 + uint8 = wraps around on overflow!
result_image = Image.fromarray(addition)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1), plt.imshow(im1arr, cmap='gray'), plt.title('Image 1'), plt.axis('off')
plt.subplot(1, 3, 2), plt.imshow(im2arr, cmap='gray'), plt.title('Image 2'), plt.axis('off')
plt.subplot(1, 3, 3), plt.imshow(addition, cmap='gray'), plt.title('Image 1 + Image 2'), plt.axis('off')
plt.show()

---
## Section 3 — Set / Logical Operations

Pixel-wise bitwise operations on two images. Lab demo: **union (OR)**.

In [ ]:
img3 = Image.fromarray((data.horse() * 255).astype(np.uint8))
img4 = Image.fromarray((data.binary_blobs(length=400) * 255).astype(np.uint8))

img3 = img3.resize(resize, Image.Resampling.LANCZOS)
img4 = img4.resize(resize, Image.Resampling.LANCZOS)

im3arr = np.asarray(img3)
im4arr = np.asarray(img4)

union = im4arr | im3arr
result_image_2 = Image.fromarray(union)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1), plt.imshow(im3arr, cmap='gray'), plt.title('Image A'), plt.axis('off')
plt.subplot(1, 3, 2), plt.imshow(im4arr, cmap='gray'), plt.title('Image B'), plt.axis('off')
plt.subplot(1, 3, 3), plt.imshow(union, cmap='gray'), plt.title('Union (A | B)'), plt.axis('off')
plt.show()

---
# Practical Tasks (solved)

## Task 1 — Change sampling and quantization parameters and observe the effects

Try multiple values of the sampling factor and the number of quantization levels to see how each affects the image.

### 1a) Effect of sampling factor (spatial resolution)

In [ ]:
factors = [2, 4, 8, 16, 32]

fig, axes = plt.subplots(1, len(factors) + 1, figsize=(20, 4))
axes[0].imshow(original_image, cmap='gray')
axes[0].set_title(f'Original {original_image.shape}'); axes[0].axis('off')

for i, f in enumerate(factors, start=1):
    s = sample_image(original_image, f)
    axes[i].imshow(s, cmap='gray')
    axes[i].set_title(f'Factor {f}\n{s.shape}'); axes[i].axis('off')

plt.tight_layout(); plt.show()

**Observation:** as the sampling factor grows, the image gets smaller and blockier — fewer pixels means less spatial detail. By factor 32 the image is unrecognizable.

### 1b) Effect of quantization levels (intensity resolution)

In [ ]:
levels_list = [2, 4, 8, 16, 64]

fig, axes = plt.subplots(1, len(levels_list) + 1, figsize=(20, 4))
axes[0].imshow(original_image, cmap='gray')
axes[0].set_title('Original (256 levels)'); axes[0].axis('off')

for i, L in enumerate(levels_list, start=1):
    q = quantize_image(original_image, L)
    axes[i].imshow(q, cmap='gray')
    axes[i].set_title(f'{L} levels'); axes[i].axis('off')

plt.tight_layout(); plt.show()

**Observation:** with only 2 levels, the image becomes pure black-and-white. With more levels (8, 16, 64), smooth gradients return. **False contours** (visible banding) appear at low levels — a classic quantization artifact.

---
## Task 2 — Image Arithmetic and Set Operations

Required operations:
1. Subtract two images and display the result
2. Add one image with a constant value 175
3. Set difference on two grayscale images
4. Symmetric difference on two grayscale images
5. Intersection on two grayscale images

### 2a) Subtract two images

In [ ]:
# Direct subtraction in uint8 can underflow (wrap around).
# Convert to int16 first, clip, then back to uint8 for a clean result.
subtraction = np.clip(im1arr.astype(np.int16) - im2arr.astype(np.int16), 0, 255).astype(np.uint8)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1), plt.imshow(im1arr, cmap='gray'), plt.title('Image 1'), plt.axis('off')
plt.subplot(1, 3, 2), plt.imshow(im2arr, cmap='gray'), plt.title('Image 2'), plt.axis('off')
plt.subplot(1, 3, 3), plt.imshow(subtraction, cmap='gray'), plt.title('Image 1 − Image 2'), plt.axis('off')
plt.show()

### 2b) Add image with constant value 175

In [ ]:
# Clip to [0, 255] to avoid uint8 overflow
added_constant = np.clip(im1arr.astype(np.int16) + 175, 0, 255).astype(np.uint8)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1), plt.imshow(im1arr, cmap='gray'), plt.title('Image 1'), plt.axis('off')
plt.subplot(1, 2, 2), plt.imshow(added_constant, cmap='gray'), plt.title('Image 1 + 175'), plt.axis('off')
plt.show()

**Notice:** the image becomes uniformly brighter. Pixels that were already above 80 saturate to white (255).

### 2c-2e) Set operations on grayscale images

| Operation | Bitwise formula | Meaning |
|---|---|---|
| **Intersection** | `A & B` | bits set in both |
| **Set difference** | `A & ~B` (A − B) | bits in A but not in B |
| **Symmetric difference** | `A ^ B` | bits in A or B but not both |
| **Union** (extra) | `A \| B` | bits in either |

In [ ]:
intersection = im3arr & im4arr
set_difference = im3arr & (~im4arr)
sym_difference = im3arr ^ im4arr
union = im3arr | im4arr

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes[0, 0].imshow(im3arr, cmap='gray'); axes[0, 0].set_title('Image A'); axes[0, 0].axis('off')
axes[0, 1].imshow(im4arr, cmap='gray'); axes[0, 1].set_title('Image B'); axes[0, 1].axis('off')
axes[0, 2].imshow(union, cmap='gray');  axes[0, 2].set_title('Union (A | B)'); axes[0, 2].axis('off')

axes[1, 0].imshow(intersection, cmap='gray');   axes[1, 0].set_title('Intersection (A & B)');         axes[1, 0].axis('off')
axes[1, 1].imshow(set_difference, cmap='gray'); axes[1, 1].set_title('Set Difference (A − B)');       axes[1, 1].axis('off')
axes[1, 2].imshow(sym_difference, cmap='gray'); axes[1, 2].set_title('Symmetric Difference (A ⊕ B)'); axes[1, 2].axis('off')

plt.tight_layout(); plt.show()

---
## Summary

| Operation | Numpy/Python | Effect |
|---|---|---|
| Sampling (down) | `cv2.resize(img, ..., INTER_NEAREST)` | Reduce spatial resolution |
| Quantization | `np.floor(img / (256//L)) * (256//L)` | Reduce intensity levels |
| Addition | `a + b` (careful: overflow!) | Brighten / blend |
| Subtraction | `np.clip(a-b, 0, 255)` | Find differences |
| Add constant | `np.clip(a+175, 0, 255)` | Brighten uniformly |
| Union | `a \| b` | Bitwise OR |
| Intersection | `a & b` | Bitwise AND |
| Set difference | `a & ~b` | A but not B |
| Symmetric diff | `a ^ b` | A or B, not both |

**Key practical tip:** When doing arithmetic on `uint8` images, **convert to int16 or float first**, then clip to `[0, 255]`, then convert back. Otherwise you get wraparound (e.g. `200 + 100 → 44` instead of `255`).

**End of notebook.**